In [ ]:
# =====================================================================
# CELL 1: ENVIRONMENT PROVISIONING & INFRASTRUCTURE SETUP
# =====================================================================
import os
import sys

# 1. Clone the GitHub repository (Replace with your actual public or private repository URL)
GIT_REPO_URL = "https://github.com/MFrizo/ufabc-pgc-wsrp.git"
REPO_NAME = "ufabc-pgc-wsrp"

if not os.path.exists(REPO_NAME):
    print(f"Cloning repository from {GIT_REPO_URL}...")
    !git clone {GIT_REPO_URL}

# 2. Change working directory to the project root
repo_path = os.path.abspath(REPO_NAME)
os.chdir(repo_path)
print(f"Current Working Directory: {os.getcwd()}")

# 3. Inject repository path into system path for modular imports
if repo_path not in sys.path:
    sys.path.append(repo_path)

# 4. Install strict dependencies from the repository's contract
print("Installing dependencies from requirements.txt...")
!pip install -r requirements.txt --quiet

# 5. Enable autoreload to dynamically fetch local code modifications
%load_ext autoreload
%autoreload 2

print("\n[SUCCESS] Environment fully configured and synchronized with GitHub.")

In [ ]:
# =====================================================================
# CELL 2: DATA INGESTION (I/O ISOLATION LAYER)
# =====================================================================
from src.utils.logger import project_logger
from src.core.data_generator import generate_wsrp_m0_instance

project_logger.info("Initializing Phase 1: Data Ingestion...")

# Generating the mock instance adhering to scientific reproducibility (seed=42)
# You can adjust num_properties according to your benchmarking scale
data_payload = generate_wsrp_m0_instance(num_properties=5, random_seed=42)

project_logger.info(f"Dataset successfully loaded.")
project_logger.info(f"Total Nodes (Depot + Properties): {data_payload['num_nodes']}")
project_logger.info(f"Coordinate matrix shape: {len(data_payload['coordinates'])}x2")

In [ ]:
# =====================================================================
# CELL 3: MATHEMATICAL MODELING & INFERENCE ENGINE (STRATEGY PATTERN)
# =====================================================================
from src.models.model_0 import build_model_m0
from src.solvers.engine import solve_model

project_logger.info("Initializing Phase 2: Building MILP Polyhedron (Pyomo - MTZ)...")
abstract_model = build_model_m0(data_payload)
project_logger.info("Polyhedron constructed successfully.")

project_logger.info("Initializing Phase 3: Dispatching to Gurobi Optimization Engine...")
# We set MIPGap to 0.0 to enforce absolute optimality proof for small instances
solved_model, benchmark_metrics = solve_model(
    model=abstract_model,
    solver_name='gurobi_direct',
    time_limit=300,
    mip_gap=0.0
)

project_logger.info(f"Solver status: {benchmark_metrics['solver_status']}")
project_logger.info(f"Total CPU Time: {benchmark_metrics['cpu_time_seconds']} seconds")

In [ ]:
# =====================================================================
# CELL 4: RESULTS PRESENTATION & GRAPH TRAVERSAL PARSING
# =====================================================================
from src.utils.parsers import print_routes

project_logger.info("Initializing Phase 4: Parsing optimized variables into human-readable routes...")

if benchmark_metrics['solver_status'] == 'ok':
    print_routes(solved_model)
else:
    project_logger.error(f"Optimization did not reach an optimal status. Termination: {benchmark_metrics['termination_condition']}")